**This notebook we use to create a variable from dev.config**

In [1]:
#import libraries

import json
import yaml
from datetime import datetime
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [2]:

# Snowflake Azure stage
stage_name = "CUSTOMER_BILLING_DB.PUBLIC.AZURE_SNOW_STAGE"

# Configuration file location
config_file = (f"@{stage_name}/config/dev_config.json")

# Processing folder date
folder_name = datetime.now().strftime("%Y%m%d")


In [3]:

# Read dev_config.json from Azure stage
rows = session.sql(f"""SELECT $1 FROM {config_file} (FILE_FORMAT => 'CUSTOMER_BILLING_DB.PUBLIC.CB_YAML_TEXT_FF')""").collect()
if not rows:
    raise Exception("Could not read dev_config.json")

config_text = "\n".join(str(row[0]) for row in rows if row[0] is not None)
config_data = json.loads(config_text)

# Declare configuration variables
env = config_data["env"]
database = config_data["database"]
input_folder = config_data["input_folder"]
unzip_folder = config_data["unzip_folder"]
expected_files = config_data["expected_files"]

control_yaml_path = config_data["control_yaml_path"]
bronze_yaml_path = config_data["bronze_yaml_path"]
silver_yaml_path = config_data["silver_yaml_path"]
gold_yaml_path = config_data["gold_yaml_path"]

dummy_date = config_data["dummy_date"]

print("Configuration loaded successfully!")
print("Environment:", env)
print("Database:", database)
print("Input folder:", input_folder)
print("Unzip folder:", unzip_folder)
